Run this file inside Google Colab, Upload the asr_segments zip file when prompted 

In [ ]:
!nvidia-smi -L
import torch; print("cuda:", torch.cuda.is_available())

In [ ]:
!pip install -q -U "transformers>=4.57.3" accelerate safetensors soundfile librosa

In [ ]:
from google.colab import files
up = files.upload()            # choose transfer/asr_segments.zip from your project
!mkdir -p /content/segments && unzip -qo /content/asr_segments.zip -d /content/segments
!ls /content/segments | wc -l

In [ ]:
import json, time
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

MODEL_ID = "b1n1yam/shook-medium-amharic-2k"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True, use_safetensors=True).to("cuda")
processor = AutoProcessor.from_pretrained(MODEL_ID)
asr = pipeline("automatic-speech-recognition", model=model,
               tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor,
               torch_dtype=torch.float16, device=0)
print("model loaded")

In [ ]:
files_ = sorted(Path("/content/segments").glob("*.wav"))
t0 = time.time()
try:
    out = asr([str(p) for p in files_], batch_size=4,
              generate_kwargs={"language": "amharic", "task": "transcribe"})
except Exception as e:
    print("language args not accepted, retrying without them:", str(e)[:150])
    out = asr([str(p) for p in files_], batch_size=4)
print(f"done in {time.time()-t0:.0f} s")

with open("/content/asr.jsonl", "w", encoding="utf-8") as f:
    for p, r in zip(files_, out):
        f.write(json.dumps({"id": p.stem, "asr_text": r["text"].strip()}, ensure_ascii=False) + "\n")
for p, r in list(zip(files_, out))[:8]:
    print(p.stem, "|", r["text"].strip())

In [ ]:
files.download("/content/asr.jsonl")